# Experiments
Each section is one experiment. Run cells top-to-bottom within a section.

**Common settings across all runs:**
- Dataset: `exp_5k` (5000 train, 700 val)
- Train metrics computed on first 12 batches only (`max_train_metric_batches=12`)
- Key metric: value **before** `train/FDE:` in progress bar = real `train/ADE` (full DDPM chain)

## Experiment 1: Deeper denoiser — `num_camlp: 4` vs `6`

**Hypothesis:** 4 CA layers bottleneck context absorption. Adding 2 more (4→6) lowers ADE  
with no architectural risk — combiner, diagonal CA, and all other settings unchanged.

In [ ]:
# Baseline: num_camlp=4
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.num_camlp=4

In [ ]:
# Treatment: num_camlp=6
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.num_camlp=6

## Experiment 2: Combiner — `TransformerContextCombiner` vs `ContextCombiner`

**Hypothesis:** `TransformerContextCombiner` adds inter-agent self-attention in the encoder,  
which should help on real multi-agent scenes even though it hurt single-sample overfitting.  
Both use zero-init so the identity-init invariant is preserved.

Config knob: `model.se_args.combiner_type` — `"transformer"` vs `"context_combiner"`.

In [ ]:
# Baseline: ContextCombiner (per-agent residual MLP, no inter-agent attention)
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner

In [ ]:
# Treatment: TransformerContextCombiner (inter-agent SA, zero-init output projections)
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=transformer

## Experiment 3: Regularization sweep — `drop_attn` and `weight_decay`

**Context:** `context_combiner` (simple per-agent residual fusion, no inter-agent attention)
beat `transformer` (adds inter-agent self-attention) on val ADE — even when the
transformer variant got bigger CA stacks. More mixing capacity didn't help and likely hurt.
On only 5000 train samples that's consistent with **overfitting**, not underfitting:
extra capacity gives the model more ways to memorize train scenes rather than generalize.

**Hypothesis:** with `combiner_type=context_combiner` fixed (the winner), increasing
regularization (`drop_attn`, `weight_decay`) should narrow the train/val ADE gap and
lower val ADE specifically — without changing any architectural capacity.

In [ ]:
# Baseline: current regularization (drop_attn=0.1 in se_args, drop_attn=0.0 in camlp/samlp, weight_decay=1e-4)
!\
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner

In [ ]:
# Treatment: stronger regularization — raise attn dropout across se/sa/ca and bump weight decay
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        model.se_args.drop_attn=0.2 \
        model.samlp_args.drop_attn=0.1 \
        model.camlp_args.drop_attn=0.1 \
        optimizer.weight_decay=0.001

## Experiment 4: Positional embedding — `lookup` vs `rope` vs `None`

**Hypothesis:** positional encoding choice is orthogonal to the combiner question and
cheap to test. `rope` often generalizes better than learned `lookup` embeddings on short
sequences with limited training data (no embedding table to overfit); `None` is a sanity
floor. Run with `combiner_type=context_combiner` (the established winner) held fixed.

Config knob: `model.se_args.pos_emb_type` — `"lookup"` / `"rope"` / `"None"`.

In [ ]:
# Baseline: lookup (current default)
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        model.se_args.pos_emb_type=lookup

In [ ]:
# Treatment: rope
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        model.se_args.pos_emb_type=rope

## Experiment 5: Self-attention depth — `num_sa_mlp: 2` vs `4`

**Hypothesis:** since adding *cross*-attention/combiner capacity (Exp 1, 2) didn't move
val ADE, the bottleneck may instead be in how well each agent's own trajectory history is
modeled *before* context is fused — i.e. self-attention depth, not cross-attention/combiner
depth. This isolates that axis: same `context_combiner`, more `num_sa_mlp` layers.

In [ ]:
# Baseline: num_sa_mlp=2 (current default)
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        model.num_sa_mlp=2

In [ ]:
# Treatment: num_sa_mlp=4
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        model.num_sa_mlp=4

## Experiment 6: Learning rate schedule — `init_value: 5e-3` vs `2e-3`

**Hypothesis:** architecture sweeps (Exp 1, 2) showed flat-to-negative returns from added
capacity — a sign the optimization itself, not the model, may be the limiting factor at
this data scale. A lower peak LR with the same cosine decay can reduce val ADE noise/
instability and improve generalization, cheaply, without touching architecture.

In [ ]:
# Baseline: init_value=5e-3 (current default)
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        scheduler.init_value=5e-3

In [ ]:
# Treatment: init_value=2e-3
!cd /home/senio/HSE/Diffusion-Trajectory-Forecaster && \
    python train.py \
        --config-name ddpm_attn \
        dataset=exp_5k \
        trainer.max_train_metric_batches=12 \
        model.se_args.combiner_type=context_combiner \
        scheduler.init_value=2e-3